In [2]:
#export MODEL="Your model here
export IMG="ghcr.io/ggml-org/llama.cpp:full-cuda"
export MODEL="unsloth/gemma-4-E4B-it-GGUF:UD-Q6_K_XL"

# podman pull $IMG # if necessary

llama-bench() {
    podman run --rm --device nvidia.com/gpu=all --entrypoint /app/llama-bench-v$(pwd)/cache:/root/.cache $IMG "$@"
}

llama-fit-params() {
    podman run --rm --device nvidia.com/gpu=all --entrypoint /app/llama-fit-params -v$(pwd)/cache:/root/.cache $IMG "$@"
}

llama-cli() {
    podman run --it --device nvidia.com/gpu=all --entrypoint /app/llama-cli -v$(pwd)/cache:/root/.cache $IMG "$@"
}


In [ ]:
# Configurable batch sizes to test to fit context
# Though it looks like batch sizes have no effects 
# as per README 
# "Increasing this value above the value of the physical batch size may improve prompt processing performance 
# when using multiple GPUs with pipeline parallelism. Default: `2048`" 

BATCH_SIZES="1024 2048" # 2 args so we can see the diff, shoud not make a difference  
UBATCH_SIZES="64 128 256 512"
FITT="256 512"

echo "Testing different batch/ubatch/fitt combinations for ${MODEL}..." 
  
for ub in $UBATCH_SIZES; do  
    for b in $BATCH_SIZES; do
        for ft in $FITT; do
            # Get fitted parameters  
            #OUTPUT=$(llama-fit-params -hf $MODEL -b $b -ub $ub | grep -o '\-c [0-9]* \-ngl [0-9]*')  \
            
            # need to add fitt for nvidia
            OUTPUT=$(llama-fit-params -hf $MODEL -b $b -ub $ub -fitt $ft 2>&1)
            
            #echo "Raw output: $OUTPUT"  # Debug line  
            
            # Extract context and GPU layers more robustly  
            CTX=$(echo "$OUTPUT" | grep -o '\-c [0-9]*' | awk '{print $2}')  
            NGL=$(echo "$OUTPUT" | grep -o '\-ngl -\?[0-9]*' | awk '{print $2}')  
            
            if [ ! -z "$CTX" ] && [ ! -z "$NGL" ]; then  
                echo "ub: $ub, b: $b, fitt: $ft, ctx: $CTX, ngl: $NGL"  
            else  
                echo "No valid parameters found"  
            fi 
        done
    done  
done 

Testing different batch/ubatch/fitt combinations for unsloth/gemma-4-E4B-it-GGUF:UD-Q6_K_XL...
ub: 64, b: 1024, fitt: 256, ctx: 33024, ngl: -1
ub: 64, b: 1024, fitt: 512, ctx: 17408, ngl: -1
ub: 64, b: 2048, fitt: 256, ctx: 33024, ngl: -1
ub: 64, b: 2048, fitt: 512, ctx: 16896, ngl: -1
ub: 128, b: 1024, fitt: 256, ctx: 27904, ngl: -1
ub: 128, b: 1024, fitt: 512, ctx: 12544, ngl: -1
ub: 128, b: 2048, fitt: 256, ctx: 28416, ngl: -1
ub: 128, b: 2048, fitt: 512, ctx: 12544, ngl: -1
ub: 256, b: 1024, fitt: 256, ctx: 19200, ngl: -1
ub: 256, b: 1024, fitt: 512, ctx: 4096, ngl: 42
ub: 256, b: 2048, fitt: 256, ctx: 19200, ngl: -1
ub: 256, b: 2048, fitt: 512, ctx: 4096, ngl: 42


In [7]:
llama-bench -hf $MODEL -ub 64,128 -p 1000,2000 -n 256,512

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 5737 MiB):
  Device 0: NVIDIA GeForce RTX 2060, compute capability 7.5, VMM: yes, VRAM: 5737 MiB
load_backend: loaded CUDA backend from /app/libggml-cuda.so
load_backend: loaded CPU backend from /app/libggml-cpu-haswell.so
| model                          |       size |     params | backend    | ngl | n_ubatch |            test |                  t/s |
| ------------------------------ | ---------: | ---------: | ---------- | --: | -------: | --------------: | -------------------: |
| gemma4 E4B Q6_K                |   6.93 GiB |     7.52 B | CUDA       |  99 |       64 |          pp1000 |      1280.21 ± 22.74 |
| gemma4 E4B Q6_K                |   6.93 GiB |     7.52 B | CUDA       |  99 |       64 |          pp2000 |      1210.74 ± 41.61 |
| gemma4 E4B Q6_K                |   6.93 GiB |     7.52 B | CUDA       |  99 |       64 |           tg256 |         50.03 ± 0.24 |
| gemma4 E4B Q6_K                |   6.93 GiB |     7.52 B | CUDA  

In [ ]:
run --bench -hf $MODEL -ub 64 -ngl 0,10,20,30,99 -p 1000,2000 -n 256,512

In [ ]:
echo "Staring llama-cli for ${MODEL}..." 
export PROMPT="Who are you?"
cli -lv 3 -hf $MODEL -ub 64 -fitt 256 --no-warmup --no-mmproj --reasoning off -ngl -1 --single-turn --prompt "$PROMPT"

In [ ]:
# nvidia server takes a while to warmup, but after that it is quick